In [1]:
USE_LOCAL_QICK = True

if USE_LOCAL_QICK:
    import sys
    import os
    print(sys.path.insert(0,"/home/larnaldi/git/6_xcom_dev/qick_lib/"))
    print(sys.path.insert(0,"/home/dmartin2/Projects/qick_internal_b_6_xcom/qick_lib"))
    print(sys.path)


# jupyter setup boilerplate
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from qick import *

# for now, all the tProc v2 classes need to be individually imported (can't use qick.*)

# the main program class
from qick.asm_v2 import AveragerProgramV2
# for defining sweeps
from qick.asm_v2 import QickSpan, QickSweep1D

print(get_version())

None
None
['/home/dmartin2/Projects/qick_internal_b_6_xcom/qick_lib', '/home/larnaldi/git/6_xcom_dev/qick_lib/', '/usr/lib64/python39.zip', '/usr/lib64/python3.9', '/usr/lib64/python3.9/lib-dynload', '', '/home/dmartin2/venv/lib64/python3.9/site-packages', '/home/dmartin2/venv/lib/python3.9/site-packages']
0.2.376


In [2]:
QICK_EMU = True
# soccfg_file_name = '/home/xilinx/jupyter_notebooks/qick/firmware/testbench/qick_testbench/soccfg.json'
soccfg_file_name = '../qick_testbench/soccfg.json'

if QICK_EMU is False:
    # soc = QickSoc('/home/xilinx/jupyter_notebooks/fw/2024-09-28_216_tprocv2r21_rfb_standard/qick_216_rfb.bit')
    # soc = QickSoc('/home/xilinx/jupyter_notebooks/fw/2025-06-15_216_tprocv2r24_standard/qick_216.bit')
    # soc = QickSoc('/home/xilinx/jupyter_notebooks/fw/qick_tprocv2_216_standard_1ch_250822_1/qick_216.bit')
    # soc = QickSoc('/home/xilinx/jupyter_notebooks/fw/qick_tprocv2_216_standard_1ch_250828_1/qick_216.bit')
    # soc = QickSoc('/home/xilinx/jupyter_notebooks/fw/qick_tprocv2_216_standard_1ch_250828_2/qick_216.bit')
    # soc = QickSoc('/home/xilinx/jupyter_notebooks/fw/qick_tprocv2_216_standard_1ch_250829_1/qick_216.bit')
    soc = QickSoc('/home/xilinx/jupyter_notebooks/fw/qick_tprocv2_216_standard_1ch_250829_2/qick_216.bit')
    soccfg = soc

    # Dump Board SoC configuration in json format and create file
    soccfg_json = soc.dump_cfg(gen_file_path=soccfg_file_name)
    # print(soccfg_json)
else:
    soc = None
    # Load soccfg.json file as a new socemucfg object
    soccfg = QickConfig(cfg=soccfg_file_name)

print(soccfg)

QICK library version mismatch: 0.2.367 remote (the board), 0.2.376 local (the PC)
                        This may cause errors, usually KeyError in QickConfig initialization.
                        If this happens, you must bring your versions in sync.


QICK running on ZCU216, software version 0.2.367

Firmware configuration (built Fri Aug 29 12:51:57 2025):

	Global clocks (MHz): tProc dispatcher timing 430.080, RF reference 245.760
	Groups of related clocks: [tProc timing clock, DAC tile 1, DAC tile 2, DAC tile 3], [DAC tile 0], [ADC tile 2]

	1 signal generator channels:
	0:	axis_signal_gen_v6 - fs=9584.640 Msps, fabric=599.040 MHz
		envelope memory: 65536 complex samples (6.838 us)
		32-bit DDS, range=9584.640 MHz
		DAC tile 0, blk 0 is 0_228 on JHC1, or QICK box DAC port 0

	1 readout channels:
	0:	axis_dyn_readout_v1 - configured by tProc output 4
		fs=2457.600 Msps, decimated=307.200 MHz, 32-bit DDS, range=2457.600 MHz
		axis_avg_buffer v1.2 (has edge counter, no weights)
		memory 8192 accumulated, 4096 decimated (13.333 us)
		triggered by tport 10, pin 0, feedback to tProc input 0
		ADC tile 2, blk 0 is 0_226 on JHC7, or QICK box ADC port 4

	8 digital output pins:
	0:	PMOD0_0_LS
	1:	PMOD0_1_LS
	2:	PMOD0_2_LS
	3:	PMOD0_3_LS
	4

In [ ]:
## Test XCOM
from qick.tprocv2_assembler import Assembler

from qick.asm_v2 import QickProgramV2

prog = QickProgramV2(soccfg)

asm = """
//TEST program

// write some registers
REG_WR r1 imm #1
REG_WR r2 imm #2    // XCOM ID in commands
REG_WR r3 imm #3
REG_WR r4 imm #424
//REG_WR r5 op -op(s_rand)

// Set number of Iterations
REG_WR r8 imm #100

TRIG p0 set

LOOP_0:

// Clear QPA data_new status bit
REG_WR s_ctrl imm ctrl_clr_qpa 
// Select QPA as Source Data
REG_WR s_cfg imm cfg_src_qpa 

// Send XCOM command when ready
WAIT qpa_rdy 
PA 6 r2 r4          // Command: Send 32bit

// Receive XCOM data
WAIT qpa_dt
WAIT time @10
// Clear QPA data_new status bit
REG_WR s_ctrl imm ctrl_clr_qpa 

//WAIT time @5
//JUMP HERE -if(S) -op(s_usr_time - #500) -uf

TRIG p0 clr

WAIT time @10

TRIG p0 set

// Set XCOM flag when ready
WAIT qpa_rdy
PA 1 r2             // Command: Set Flag

// Select QPA as Source of Flag
REG_WR s_cfg imm cfg_flg_qpa

// Wait for Peripheral Flag
JUMP HERE -if(NF)

TRIG p0 clr

WAIT time @20

TRIG p0 set

// Clear XCOM flag
WAIT qpa_rdy
PA 0 r2           // Command: Clear Flag

// Select QPA as Source of Flag
REG_WR s_cfg imm cfg_flg_qpa

// Wait for Peripheral Flag
JUMP HERE -if(F)

TRIG p0 clr

// Increase dmem iterations counter
REG_WR r10 dmem [&0]
DMEM_WR [&0] op -op(r10 + #1)

// Main Loop Iterations
TEST -op(r8 - #0)
JUMP LOOP_0 -if(NZ) -wr(r8 op) -op(r8 - #1)

TRIG p0 clr

// Save Elapsed Time -> USR Time
REG_WR r9 op -op(s_usr_time)

.END
"""

# Compile program and generate PMEM content
p_list, label_dict = Assembler.str_asm2list(asm)
prog.labels = label_dict
prog.prog_list = p_list
prog._make_binprog()

# print(prog)
prog.print_pmem2hex()

// PMEM content
000000000000000000
8c60000000000000a1
8c6000000000000122
8c60000000000001a3
8c600000000000d424
8c6000000000003228
dd8000300000000000
8c6000000008000002
8c6000000000000202
081200000500008000
389201200500008000
644600001112000000
081200000500010000
389201800500010000
081100000580000000
391101c00580000000
8c6000000008000002
dd8000100000000000
081100000580000000
391102400580000000
dd8000300000000000
081200000500008000
389202a00500008000
644100001100000000
8c6000000000003802
3f0003200000000000
dd8000100000000000
081100000580000500
391103600580000500
dd8000300000000000
081200000500008000
389203c00500008000
644000001100000000
8c6000000000003802
3e8004400000000000
dd8000100000000000
9c200000000000002a
b80000001500000080
081100001400000000
398900e014000000a8
dd8000100000000000
840000000580000029
3c0005400000000000
